In [0]:
# =============================================================================
# Notebook: 04_bronze_to_silver_cdc
# Purpose : Move order data from Bronze to Silver layer.
#           First run = full load. Later runs = CDC merge (insert/update).
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

# ---- Storage paths (ADLS Gen2 via Unity Catalog External Locations) ----
storage_account = "stretailcdcproj"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail_orders/"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/retail_orders/"

# ---- Step 1: Read raw data from Bronze ----
df_bronze = spark.read.format("delta").load(bronze_path)
print(f"Rows in bronze: {df_bronze.count()}")

# ---- Step 2: Keep only the latest record per order_id ----
# Same order_id can appear more than once (updates/corrections).
# We rank by last_modified and keep the most recent version only.
window_spec = Window.partitionBy("order_id").orderBy(F.col("last_modified").desc())

df_deduped = (
    df_bronze
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
print(f"Rows after dedup: {df_deduped.count()}")

# ---- Step 3: Flag missing amount instead of dropping/filling it ----
# Keeps the row but marks it, so bad data isn't hidden.
df_cleaned = df_deduped.withColumn(
    "is_amount_missing",
    F.when(F.col("amount").isNull(), F.lit(True)).otherwise(F.lit(False))
)

# ---- Step 4: Add a timestamp for when this record was processed ----
# Useful for tracking/debugging pipeline runs later.
df_cleaned = df_cleaned.withColumn("silver_load_timestamp", F.current_timestamp())

# ---- Step 5: Load into Silver — full load if new, else CDC merge ----
# Checks if Silver table already exists:
#   - No  -> first run, write full data
#   - Yes -> apply MERGE: update existing orders, insert new ones
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Silver table not found — running FULL LOAD.")
    df_cleaned.write.mode("overwrite").format("delta").save(silver_path)
    print(f"Full load complete: {df_cleaned.count()} rows written")

else:
    print("Silver table exists — running CDC MERGE.")
    silver_table = DeltaTable.forPath(spark, silver_path)

    (
        silver_table.alias("target")
        .merge(df_cleaned.alias("source"), "target.order_id = source.order_id")
        .whenMatchedUpdateAll()      # same order_id -> update with latest data
        .whenNotMatchedInsertAll()   # new order_id -> insert as new row
        .execute()
    )

    updated_count = spark.read.format("delta").load(silver_path).count()
    print(f"CDC merge complete. Silver now has {updated_count} rows")

df_cleaned.show(5)

# ---- Step 6: Check no duplicate order_id remains ----
final_df = spark.read.format("delta").load(silver_path)
total_rows = final_df.count()
distinct_orders = final_df.select("order_id").distinct().count()

print(f"\nCheck: total_rows={total_rows}, distinct_order_ids={distinct_orders}")
if total_rows == distinct_orders:
    print("No duplicate order_id in Silver layer")
else:
    print("Duplicate order_id found — needs investigation")


Rows in bronze: 5170
Rows after dedup: 5100
Silver table exists — running CDC MERGE.
CDC merge complete. Silver now has 5100 rows
+--------+-----------+-----------+------+--------+------+-------------------+-------------------+-----------------+---------------------+
|order_id|customer_id|    product|region|quantity|amount|         order_date|      last_modified|is_amount_missing|silver_load_timestamp|
+--------+-----------+-----------+------+--------+------+-------------------+-------------------+-----------------+---------------------+
|       1|       1754|    Shampoo| North|       5|115.49|2024-02-27 00:00:00|2024-01-13 23:00:00|            false| 2026-09-11 21:51:...|
|       2|       1616|    Shampoo| South|       5|120.17|2024-08-04 00:00:00|2024-01-17 00:00:00|            false| 2026-09-11 21:51:...|
|       3|       1006|Conditioner|  West|       3|405.67|2024-08-02 00:00:00|2024-04-22 14:00:00|            false| 2026-09-11 21:51:...|
|       4|       1094|   Lipstick| North| 